# Etapa 5: treino do GPT com BPE no Colab

Este notebook treina dois GPTs (pequeno, ~4,3 M de parâmetros (parameters), e médio, ~12 M)
com o tokenizador (tokenizer) BPE da etapa 4, na GPU gratuita do Colab, e compara os dois
com o GPT por caracteres da etapa 3 em **bits por byte**.

O notebook é fino de propósito: toda a lógica está no pacote `machado_gpt`, que tem testes.
Aqui ficam só os passos, e o **porquê** de cada um. O diário completo está em
`docs/etapas/05-treino-colab.md`.

**Antes de começar:** menu *Ambiente de execução → Alterar o tipo de ambiente de
execução → GPU T4*. Sem isso, o treino roda na CPU e leva horas.

**Não salve este notebook com as saídas (outputs) de volta no GitHub:** elas teriam
textos gerados e caminhos do seu Drive.

## 1. Conferir a GPU

**Por quê:** a GPU faz milhares de multiplicações em paralelo. O treino é quase só
multiplicação de matrizes, então ela é dezenas de vezes mais rápida que a CPU da etapa 3.
A T4 tem 16 GB de memória, de sobra para modelos deste tamanho.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Sem GPU: troque o tipo de ambiente de execução para T4"
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name())

## 2. Baixar o código

**Por quê:** o repositório é privado, então o Colab precisa de uma credencial para lê-lo.
Usamos um token do GitHub *fine-grained*, só de leitura e só deste repositório, guardado
nos **Segredos** do Colab (ícone de chave na barra lateral), com o nome `GITHUB_TOKEN`.
O token nunca aparece no notebook nem fica gravado no `.git/config`: ele vai num
cabeçalho (header) passado só para o comando `git clone`.

Preencha `REPO` com `seu-usuario/machado-gpt`. `BRANCH` é a versão do código.

In [ ]:
REPO = ""  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

import base64, os, subprocess
from google.colab import userdata

assert REPO, "Preencha REPO (por exemplo: seu-usuario/machado-gpt)"
ROOT = "/content/machado-gpt"
if not os.path.exists(ROOT):
    auth = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()
    result = subprocess.run(
        ["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}",
         "clone", "--quiet", "--branch", BRANCH, f"https://github.com/{REPO}.git", ROOT],
        capture_output=True, text=True)
    del auth
    assert result.returncode == 0, "git clone falhou: confira REPO, BRANCH e o segredo GITHUB_TOKEN"
%cd {ROOT}
!git log --oneline -1

## 3. Instalar as dependências nas versões exatas

**Por quê:** o Colab já tem PyTorch (com GPU) e NumPy. Falta o `pdfplumber`, que extrai o
texto dos PDFs. Instalamos **a mesma versão** usada localmente (a do `uv.lock`): outra
versão poderia ler um espaço ou uma quebra de linha de outro jeito, e o corpus mudaria.
O passo 5 confere que não mudou.

Em vez de instalar o pacote, apontamos o Python para a pasta `src/` (`PYTHONPATH`), para
não mexer no PyTorch do Colab.

In [ ]:
!pip install --quiet pdfplumber==0.11.10 pdfminer.six==20260107
%env PYTHONPATH=/content/machado-gpt/src
import sys
sys.path.insert(0, "/content/machado-gpt/src")

## 4. Regenerar o corpus e os símbolos (tokens) aqui mesmo

**Por quê:** o corpus e os arquivos codificados nunca entram no Git nem no Drive. Em vez de
copiá-los, refazemos o caminho inteiro das etapas 1 e 4:

1. **download:** baixa os PDFs do commit fixado do luxedo, conferindo o hash de cada um;
2. **extração (extraction):** aplica as regras da etapa 1 a cada PDF (a parte mais
   demorada: 10 a 20 min, porque a CPU do Colab tem só 2 núcleos);
3. **construção (build):** deduplicação e divisão (split) por obra;
4. **tokenização:** treina o BPE só no treino e codifica os três conjuntos.

Tudo é determinístico, então o resultado deve ser **idêntico, byte a byte**, ao da sua
máquina. Isso também testa a reprodutibilidade do projeto.

In [ ]:
!python -m machado_gpt.corpus.download

In [ ]:
from pathlib import Path
from machado_gpt.corpus import extract

names = sorted(str(p.relative_to(extract.PDF_DIR)) for p in extract.PDF_DIR.glob("*/*.pdf"))
print(len(names), "PDFs")
extract.main(names)

In [ ]:
!python -m machado_gpt.corpus.build
!python -m machado_gpt.tokenize_corpus

## 5. Conferir que é o mesmo corpus

**Por quê:** "deve ser idêntico" não basta: medimos.

- O catálogo `data/catalog/works.csv` tem o SHA-256 do texto limpo de cada obra. Se a
  extração no Colab desse um só caractere diferente, o `git diff` mostraria a linha.
- O `results/04-bpe.json` regenerado tem os SHA-256 do tokenizador e dos três conjuntos.
  Comparamos com a versão do Git e depois restauramos a versão do Git, que tem também o
  estudo de tamanhos.

O script de treino confere esses hashes de novo antes de começar, e se recusa a treinar se
algo diferir.

In [ ]:
import json, subprocess

diff = subprocess.run(["git", "diff", "--stat", "data/catalog/"], capture_output=True, text=True).stdout
assert not diff, "o catálogo mudou: a extração no Colab difere da local\n" + diff
new = json.load(open("results/04-bpe.json"))
old = json.loads(subprocess.run(["git", "show", "HEAD:results/04-bpe.json"], capture_output=True, text=True).stdout)
assert new["tokenizer_sha256"] == old["tokenizer_sha256"], "tokenizador diferente"
for name in ("train", "val", "test"):
    assert new["splits"][name]["sha256"] == old["splits"][name]["sha256"], f"{name} diferente"
!git checkout -- results/04-bpe.json
print("corpus, tokenizador e conjuntos idênticos aos da etapa 4")

## 6. Conectar o Google Drive para os checkpoints

**Por quê:** tudo em `/content` some quando a sessão do Colab termina. A sessão gratuita cai
por inatividade ou depois de algumas horas. Os checkpoints vão para uma pasta no seu Drive:

- `best.pt`: o melhor modelo pela validação (validation), o que vale no fim;
- `last.pt`: o estado completo do treino (pesos (weights), otimizador (optimizer), estados
  aleatórios), salvo a cada 500 passos, para retomar se a sessão cair.

O Colab vai pedir autorização para acessar o Drive da conta em que você está logado.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/machado-gpt"

## 7. Treinar o modelo pequeno

**O que acontece em cada passo (step):** sorteamos 64 janelas (windows) de 256 símbolos do
treino, o modelo prevê o próximo símbolo em cada posição, calculamos a perda (loss) e
ajustamos os pesos. É o mesmo laço (loop) da etapa 3, com três adições para a GPU:

- **precisão mista (mixed precision):** as contas pesadas em float16, com metade da memória
  e cerca do dobro da velocidade. O softmax e a perda ficam em float32. Um *gradient scaler*
  multiplica a perda antes do backward para que gradientes pequenos não virem zero em
  float16;
- **avaliação exata:** a cada 100 passos, a perda sobre a validação **inteira**;
- **parada antecipada (early stopping):** se a validação não melhora por 10 avaliações
  seguidas (1.000 passos), o treino para. O `best.pt` guarda o melhor ponto.

**O que observar:** no começo, treino e validação caem juntos. Quando o modelo começa a
decorar o treino (sobreajuste, overfitting), o treino continua caindo e a validação para de
cair, ou sobe.

**Se a sessão cair:** rode os passos 1 a 6 de novo e depois esta célula. O `--resume`
continua do último `last.pt`, com o mesmo resultado (há um teste para isso).

In [ ]:
!python -m machado_gpt.train_bpe --size small --out {CKPT}/05-small --resume

## 8. Treinar o modelo médio

**Por quê:** o médio tem ~3 vezes mais parâmetros (6 camadas (layers), C = 384). A pergunta
do experimento: com só 2,25 milhões de símbolos de treino, a capacidade extra melhora a
validação ou só faz o modelo decorar mais cedo? A taxa de aprendizado (learning rate) é
menor (6·10⁻⁴), porque modelos maiores ficam instáveis com passos grandes.

In [ ]:
!python -m machado_gpt.train_bpe --size medium --out {CKPT}/05-medium --resume

## 9. Comparar as curvas

**Por quê:** os números finais dizem *quem* ganhou. As curvas mostram *por quê*: em que
passo cada modelo começou a decorar e qual a distância entre treino e validação.

As perdas estão em nats por símbolo. Para comparar com a etapa 3 (caracteres), a tabela usa
**bits por byte**: o total de informação dividido pelo número de bytes do texto, que não
depende de como o texto foi cortado em símbolos.

In [ ]:
import json
import matplotlib.pyplot as plt

runs = {size: json.load(open(f"results/05-gpt-bpe-{size}.json")) for size in ("small", "medium")}
fig, ax = plt.subplots(figsize=(9, 5))
for (size, r), color in zip(runs.items(), ("tab:blue", "tab:orange")):
    steps = [p["step"] for p in r["history"]]
    ax.plot(steps, [p["train"] for p in r["history"]], "--", color=color, label=f"{size}: treino")
    ax.plot(steps, [p["val"] for p in r["history"]], "-", color=color, label=f"{size}: validação")
    ax.axvline(r["best_step"], color=color, alpha=0.3)
ax.set_xlabel("passo"); ax.set_ylabel("perda (nats por símbolo)"); ax.set_ylim(top=6); ax.legend(); ax.grid(alpha=0.3)
plt.show()

print(f"{'modelo':22} {'parâmetros':>11} {'melhor passo':>13} {'val bits/byte':>14} {'teste bits/byte':>16}")
print(f"{'bigrama (etapa 2)':22} {'':>11} {'':>13} {3.33:14.2f}")
print(f"{'GPT caracteres (et. 3)':22} {846_592:11,} {3000:13} {2.22:14.2f}")
for size, r in runs.items():
    print(f"{'GPT BPE ' + size:22} {r['parameters']:11,} {r['best_step']:13} "
          f"{r['val']['bits_per_byte']:14.3f} {r['test']['bits_per_byte']:16.3f}")

## 10. Trazer os resultados de volta

**Por quê:** os `results/05-gpt-bpe-*.json` (métricas, curvas, amostras (samples) e o
SHA-256 de cada `best.pt`) são versionados no Git; os pesos, não. Baixe os dois arquivos e
coloque-os na pasta `results/` do projeto local. Há também uma cópia de cada um no Drive,
ao lado do checkpoint.

In [ ]:
from google.colab import files
for size in ("small", "medium"):
    files.download(f"results/05-gpt-bpe-{size}.json")